# Malaria Detection — Disease Detection Framework, Module 1

Binary classification of blood smear cell images (Parasitized vs Uninfected) using transfer learning.

Dataset: https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria
Add it via **Add Input** in the Kaggle notebook sidebar before running.

In [ ]:
!pip install -q timm albumentations grad-cam
import os, copy, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, classification_report
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = "/kaggle/input/cell-images-for-detecting-malaria/cell_images"
CLASS_NAMES = ["Parasitized", "Uninfected"]
BACKBONE = "efficientnet_b3"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 5
EPOCHS_FINETUNE = 20
LR_HEAD = 1e-3
LR_FINETUNE = 3e-5
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
WEIGHTS_PATH = "malaria_best.pt"

## 1. Load and explore the data

In [ ]:
from pathlib import Path

def build_from_imagefolder(root_dir, class_names):
    root = Path(root_dir)
    records = []
    for label_idx, class_name in enumerate(class_names):
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"):
            for filepath in class_dir.glob(ext):
                records.append({"filepath": str(filepath), "label": label_idx})
    return pd.DataFrame(records)

df = build_from_imagefolder(DATA_DIR, CLASS_NAMES)
print("total images:", len(df))
print(df["label"].value_counts().rename(index={i: n for i, n in enumerate(CLASS_NAMES)}))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, label_idx in enumerate([0, 1]):
    samples = df[df["label"] == label_idx].sample(4, random_state=SEED)
    for row, (_, sample_row) in enumerate(samples.iterrows()):
        img = cv2.cvtColor(cv2.imread(sample_row["filepath"]), cv2.COLOR_BGR2RGB)
        axes[col, row].imshow(img)
        axes[col, row].set_title(CLASS_NAMES[label_idx])
        axes[col, row].axis("off")
plt.tight_layout()
plt.show()

## 2. Train / validation / test split

In [ ]:
train_val_df, test_df = train_test_split(df, test_size=TEST_SPLIT, stratify=df["label"], random_state=SEED)
val_relative = VAL_SPLIT / (1 - TEST_SPLIT)
train_df, val_df = train_test_split(train_val_df, test_size=val_relative, stratify=train_val_df["label"], random_state=SEED)
print("train / val / test sizes:", len(train_df), len(val_df), len(test_df))

class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(len(CLASS_NAMES)), y=train_df["label"].values)
class_weights = class_weights.astype("float32")
print("class weights:", class_weights)

## 3. Data pipeline

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-20, 20), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.GaussNoise(p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CoarseDropout(p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

class MedicalImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.cvtColor(cv2.imread(str(row["filepath"])), cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, int(row["label"])

train_ds = MedicalImageDataset(train_df, get_train_transforms(IMG_SIZE))
val_ds = MedicalImageDataset(val_df, get_val_transforms(IMG_SIZE))
test_ds = MedicalImageDataset(test_df, get_val_transforms(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

## 4. Model

In [ ]:
def build_model(backbone_name, num_classes, pretrained=True):
    return timm.create_model(backbone_name, pretrained=pretrained, num_classes=num_classes)

def freeze_backbone(model):
    classifier = model.get_classifier()
    classifier_param_ids = set(id(p) for p in classifier.parameters())
    for param in model.parameters():
        if id(param) not in classifier_param_ids:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

model = build_model(BACKBONE, len(CLASS_NAMES), pretrained=True).to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

## 5. Train — stage 1 (head only), stage 2 (full fine-tune)

In [ ]:
def run_training_stage(model, epochs, lr, patience=6):
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=0.1)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    use_amp = DEVICE.type == "cuda"
    scaler = GradScaler(enabled=use_amp)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast(device_type=DEVICE.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                with autocast(device_type=DEVICE.type, enabled=use_amp):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                correct += (outputs.argmax(dim=1) == labels).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch+1}/{epochs}  train_loss {train_loss:.4f}  val_loss {val_loss:.4f}  val_acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return history, best_val_acc

In [ ]:
print("stage 1: training classifier head only")
freeze_backbone(model)
history_head, _ = run_training_stage(model, EPOCHS_HEAD, LR_HEAD)

In [ ]:
print("stage 2: fine-tuning full network")
unfreeze_all(model)
history_finetune, best_val_acc = run_training_stage(model, EPOCHS_FINETUNE, LR_FINETUNE)
print("best validation accuracy:", best_val_acc)

## 6. Evaluate on the held-out test set

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        probs = torch.softmax(model(images), dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels, all_preds, all_probs = np.array(all_labels), np.array(all_preds), np.array(all_probs)

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
auc = roc_auc_score(all_labels, all_probs[:, 1])

print(f"test accuracy:  {accuracy:.4f}")
print(f"precision (macro): {precision:.4f}")
print(f"recall (macro):    {recall:.4f}")
print(f"f1 (macro):        {f1:.4f}")
print(f"auc-roc:           {auc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES)
ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 7. Grad-CAM — verify the model looks at the actual parasite, not background

In [ ]:
def get_target_layers(model, backbone_name):
    name = backbone_name.lower()
    if "efficientnet" in name:
        return [model.conv_head]
    if "resnet" in name or "resnext" in name:
        return [model.layer4[-1]]
    if "convnext" in name:
        return [model.stages[-1].blocks[-1]]
    if "densenet" in name:
        return [model.features.norm5]
    return [list(model.children())[-3]]

target_layers = get_target_layers(model, BACKBONE)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_indices = np.random.choice(len(test_ds), 4, replace=False)
for ax, idx in zip(axes, sample_indices):
    image_tensor, label = test_ds[idx]
    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    rgb_orig = cv2.cvtColor(cv2.imread(test_df.iloc[idx]["filepath"]), cv2.COLOR_BGR2RGB)
    rgb_resized = cv2.resize(rgb_orig, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0

    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(label)])[0]
    visualization = show_cam_on_image(rgb_resized, grayscale_cam, use_rgb=True)

    ax.imshow(visualization)
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Export trained weights

In [ ]:
torch.save(model.state_dict(), WEIGHTS_PATH)
print(f"saved to {WEIGHTS_PATH} — download this file from the notebook output panel")
print("place it in model_weights/malaria_best.pt in the backend project")